# Clinical exploratory analysis: sliding-window design (v3)

Exploratory analysis of the clinical and drug variables and of the outcomes, on the same analytic cohort as the HRV EDA (windows with >= 6 h of valid HRV).

Note: this notebook was run on `ultimate_dataset_v3.csv`. That file differs from `ultimate_dataset_v3_1.csv` only in the LF, HF and LF/HF features (not log-transformed), which this notebook does not use except for the HRV comparison in C6.

| Section | Question |
|---|---|
| C1 | Cohort characteristics (Table 1, patient level) |
| C2 | Missing data per clinical variable |
| C3 | Physiological plausibility |
| C4 | Redundancy among clinical variables |
| C5 | Included versus excluded windows |
| C6 | How well clinical variables separate each outcome, compared with HRV |
| C7 | Construction of the SOFA outcome (components, delta SOFA, new laboratory values) |
| C8 | Construction of the haemodynamic outcome (MAP and lactate criteria) |
| C9 | Clinical trajectories over the stay |
| C10 | Events per variable for each feature set (overfitting risk) |

**Rules** (same as the HRV EDA): descriptive only; comparisons use the SMD, not p-values (|SMD| < 0.1 negligible; 0.1-0.2 small; > 0.2 relevant); the dataset is not modified. Outputs are written to `eda_clinico_v3/`.

In [ ]:

# 0. Setup and data loading
import os, json, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
warnings.filterwarnings('ignore')
sns.set_style('whitegrid')
pd.set_option('display.max_columns', None)

try:
    from google.colab import drive
    drive.mount('/content/drive')
except ImportError:
    pass
try:
    display
except NameError:
    display = print

RUTA_BASE = '/content/drive/MyDrive/tesis_sepsis'
RUTA_ULTIMATE = f'{RUTA_BASE}/ultimate_dataset_v3.csv'
RUTA_COLUMNAS = f'{RUTA_BASE}/columnas_v3.json'
RUTA_REPORTE = f'{RUTA_BASE}/df_reporte_final_unificado.csv'
RUTA_SMD_HRV = f'{RUTA_BASE}/eda_hrv_v3/B6_smd_por_label.csv'  # from the HRV EDA (optional)
RUTA_EDA = f'{RUTA_BASE}/eda_clinico_v3'
os.makedirs(RUTA_EDA, exist_ok=True)
UMBRAL_HRV = 6

LABELS = {
    'label_sofa_responder':      ('SOFA respuesta (principal)',     {1: 'Responder', 0: 'No responder'}),
    'label_cardio':              ('Hemodinámico (principal)',       {1: 'Responder', 0: 'No responder'}),
    'label_sofa_deterioro_2pts': ('SOFA progresión ≥2 (secundario)', {1: 'Progresión', 0: 'Sin progresión'}),
}

ult = pd.read_csv(RUTA_ULTIMATE, parse_dates=['t0_prediccion', 'outtime', 'deathtime'])
with open(RUTA_COLUMNAS) as f:
    COLS = json.load(f)

# Derived variables for description only (not added to the dataset)
ult['sexo_masculino'] = ult['gender'].astype(str).str.upper().str.startswith('M').astype(int)
ult['vasopresor_activo'] = (ult['ned'].fillna(0) > 0).astype(int)
ult['lactato_medido_12h'] = ult['lactate_last_actual12h'].notna().astype(int)

dfv = ult[ult[f'incl_hrv_{UMBRAL_HRV}h'] == 1].copy()  # analytic cohort

# Clinical variables used by the models (from columnas_v3.json)
CLIN = [c for c in COLS['features_clinicas'] if c != 'gender'] + ['sexo_masculino']
SOFA_F = COLS['features_sofa']
PAMLAC = COLS['features_pam_lactato_actual']
CLIN_MODELO = CLIN + PAMLAC + SOFA_F
BINARIAS = ['sexo_masculino', 'sedante_bin', 'opioide_bin', 'vasopresor_activo', 'lactato_medido_12h']
CONTINUAS = [c for c in CLIN_MODELO if c not in BINARIAS]
COMPONENTES = ['sofa_resp_t0', 'sofa_coag_t0', 'sofa_liver_t0', 'sofa_cardio_t0', 'sofa_neuro_t0', 'sofa_renal_t0']

NOMBRES = {
    'age': 'Edad (años)', 'sexo_masculino': 'Sexo masculino', 'map_t0': 'PAM en T0 (mmHg)',
    'rr_t0': 'Frecuencia respiratoria (rpm)', 'spo2_t0': 'SpO₂ (%)', 'temp_t0': 'Temperatura (°C)',
    'ned': 'NED en T0 (µg/kg/min)', 'vis': 'VIS en T0', 'cumvp': 'cumVP (12 h)',
    'sedante_bin': 'Sedación (12 h)', 'opioide_bin': 'Opioides (12 h)',
    'mbp_mean_actual12h': 'PAM media 12 h (mmHg)', 'lactate_last_actual12h': 'Último lactato 12 h (mmol/L)',
    'sofa_t0_parcial': 'SOFA en T0 (componentes disponibles)', 'n_comp_validos_t0': 'Componentes SOFA medidos',
    'vasopresor_activo': 'Vasopresor activo en T0', 'lactato_medido_12h': 'Lactato medido en 12 h',
}

reporte = pd.read_csv(RUTA_REPORTE) if os.path.exists(RUTA_REPORTE) else None
print(f"Ventanas candidatas : {len(ult)} | pacientes {ult.icustay_id.nunique()}")
print(f"Cohorte analítica   : {len(dfv)} | pacientes {dfv.icustay_id.nunique()}")
print(f"Variables clínicas del modelo ({len(CLIN_MODELO)}): {CLIN_MODELO}")


In [ ]:

# Shared helpers
def smd(a, b, binaria=False):
    a, b = pd.Series(a).dropna(), pd.Series(b).dropna()
    if len(a) < 2 or len(b) < 2:
        return np.nan
    if binaria:
        p1, p0 = a.mean(), b.mean()
        den = np.sqrt((p1 * (1 - p1) + p0 * (1 - p0)) / 2)
        return (p1 - p0) / den if den > 0 else np.nan
    sp = np.sqrt((a.var() + b.var()) / 2)
    return (a.mean() - b.mean()) / sp if sp > 0 else np.nan

def mediana_riq(s):
    s = s.dropna()
    if not len(s):
        return '—'
    d = 2 if abs(s.median()) < 1 else 1  # more decimals for small values (NED)
    return f"{s.median():.{d}f} [{s.quantile(.25):.{d}f}–{s.quantile(.75):.{d}f}]"

def n_pct(s):
    s = s.dropna()
    return f"{int(s.sum())} ({100 * s.mean():.1f} %)" if len(s) else '—'

def colorear_smd(v):
    if pd.isna(v): return ''
    return 'background-color:#f4cccc' if abs(v) > .2 else ('background-color:#fff2cc' if abs(v) > .1 else '')


---
## C1. Table 1: cohort characteristics (patient level)

Each patient is described at the **first analytic window** (earliest T0 with >= 6 h of HRV). Mortality is in-hospital (`deathtime` of the admission).

In [ ]:

print("=" * 70); print("C1. TABLA 1 (primera ventana analítica de cada paciente)"); print("=" * 70)
primera = dfv.sort_values(['icustay_id', 'hour_index']).groupby('icustay_id').first().reset_index()
vpp = dfv.groupby('icustay_id').size()

filas = [('Pacientes, n', f"{len(primera)}"),
         ('Ventanas analíticas por paciente', mediana_riq(vpp)),
         ('Hora del primer T0 desde el inicio', mediana_riq(primera['hour_index'])),
         ('Edad (años)', mediana_riq(primera['age'])),
         ('Sexo masculino', n_pct(primera['sexo_masculino']))]
if reporte is not None and 'tipo_t0' in reporte.columns:
    tipos = primera.merge(reporte[['icustay_id', 'tipo_t0']].drop_duplicates('icustay_id'), on='icustay_id', how='left')['tipo_t0']
    for t, n in tipos.value_counts().items():
        filas.append((f'Inicio de sepsis: {t}', f"{n} ({100 * n / len(primera):.1f} %)"))
for c in ['sofa_t0_parcial', 'n_comp_validos_t0', 'map_t0', 'mbp_mean_actual12h', 'rr_t0', 'spo2_t0']:
    filas.append((NOMBRES[c], mediana_riq(primera[c])))
filas.append(('Lactato medido en las 12 h previas', n_pct(primera['lactato_medido_12h'])))
filas.append(('Último lactato (si se midió, mmol/L)', mediana_riq(primera['lactate_last_actual12h'])))
filas.append(('Vasopresor activo en T0', n_pct(primera['vasopresor_activo'])))
filas.append(('NED en T0 (si > 0, µg/kg/min)', mediana_riq(primera.loc[primera['ned'] > 0, 'ned'])))
filas.append(('Sedación en las 12 h previas', n_pct(primera['sedante_bin'])))
filas.append(('Opioides en las 12 h previas', n_pct(primera['opioide_bin'])))
filas.append(('Mortalidad hospitalaria', n_pct(primera['deathtime'].notna().astype(int))))

tabla1 = pd.DataFrame(filas, columns=['Variable', 'Valor'])
display(tabla1)
tabla1.to_csv(f'{RUTA_EDA}/C1_tabla1.csv', index=False)
print("Continuas: mediana [RIQ]. Categóricas: n (%).")


## C2. Missing data in the analytic cohort

Description only. Missingness in clinical variables is rarely random (for example, lactate is ordered when hypoperfusion is suspected).

In [ ]:

print("=" * 70); print("C2. DATOS FALTANTES (ventanas analíticas)"); print("=" * 70)
vars_nulos = CLIN_MODELO + ['temp_t0'] + COMPONENTES
nul = pd.DataFrame({'variable': [NOMBRES.get(c, c) for c in vars_nulos],
                    'columna': vars_nulos,
                    'pct_faltante': [round(dfv[c].isna().mean() * 100, 1) for c in vars_nulos]}).sort_values('pct_faltante', ascending=False)
display(nul.reset_index(drop=True))
nul.to_csv(f'{RUTA_EDA}/C2_faltantes.csv', index=False)

plt.figure(figsize=(9, 6))
sns.barplot(data=nul, x='pct_faltante', y='variable', color='indianred')
plt.xlabel('% de ventanas sin dato'); plt.ylabel(''); plt.title('Faltantes por variable (cohorte analítica)')
plt.tight_layout(); plt.savefig(f'{RUTA_EDA}/C2_faltantes.png', dpi=150); plt.show()

print("\nLactato medido en las 12 h previas según el desenlace hemodinámico sea evaluable:")
display(dfv.groupby(dfv['label_cardio'].notna().map({True: 'Evaluable', False: 'No evaluable'}))['lactato_medido_12h'].mean().mul(100).round(1).to_frame('% con lactato'))
print("La temperatura (temp_t0) se describe pero no entra a los modelos.")


## C3. Distributions and physiological plausibility

**Statistical outliers** (IQR) may be real severe patients and are not removed; values **outside the physiological range** would indicate recording errors.

In [ ]:

print("=" * 70); print("C3. DISTRIBUCIONES Y PLAUSIBILIDAD"); print("=" * 70)
RANGOS = {'age': (16, 100), 'map_t0': (20, 200), 'mbp_mean_actual12h': (20, 200), 'rr_t0': (0, 80),
          'spo2_t0': (50, 100), 'temp_t0': (30, 43), 'lactate_last_actual12h': (0, 30),
          'ned': (0, 5), 'sofa_t0_parcial': (0, 24), 'n_comp_validos_t0': (0, 6)}
filas = []
for c in CONTINUAS + ['temp_t0']:
    s = dfv[c].dropna()
    if len(s) < 5: continue
    q1, q3 = s.quantile([.25, .75]); iqr = q3 - q1
    lo, hi = RANGOS.get(c, (-np.inf, np.inf))
    filas.append({'variable': NOMBRES.get(c, c), 'n': len(s), 'mediana': round(s.median(), 2),
                  'min': round(s.min(), 2), 'max': round(s.max(), 2), 'asimetria': round(s.skew(), 2),
                  'pct_atipicos_IQR': round(100 * ((s < q1 - 1.5 * iqr) | (s > q3 + 1.5 * iqr)).mean(), 1),
                  'n_fuera_de_rango': int(((s < lo) | (s > hi)).sum())})
desc = pd.DataFrame(filas)
display(desc); desc.to_csv(f'{RUTA_EDA}/C3_distribuciones.csv', index=False)
if desc['n_fuera_de_rango'].sum() == 0:
    print("✅ Ningún valor fuera de rango fisiológico.")
else:
    print("⚠️  Hay valores fuera de rango: revisar en la fuente (no se eliminan aquí).")

cont = [c for c in CONTINUAS if dfv[c].notna().sum() > 5]
ncol = 4; nrow = int(np.ceil(len(cont) / ncol))
fig, axes = plt.subplots(nrow, ncol, figsize=(18, 3.6 * nrow)); axes = np.atleast_1d(axes).flatten()
for ax, c in zip(axes, cont):
    s = dfv[c].dropna()
    if c in ('ned', 'vis', 'cumvp'):
        s = s[s > 0]; ax.set_title(f'{NOMBRES.get(c, c)} (solo > 0)', fontsize=10)
    else:
        ax.set_title(NOMBRES.get(c, c), fontsize=10)
    sns.histplot(s, bins=35, ax=ax, color='teal'); ax.set_xlabel('')
for ax in axes[len(cont):]: fig.delaxes(ax)
plt.tight_layout(); plt.savefig(f'{RUTA_EDA}/C3_distribuciones.png', dpi=150); plt.show()

print("Binarias (% de ventanas):")
display(pd.Series({NOMBRES[c]: round(100 * dfv[c].mean(), 1) for c in BINARIAS}).to_frame('% de ventanas'))


## C4. Redundancy among clinical variables

Pairs with |rho| >= 0.9 carry almost the same information (expected: NED and VIS, MAP at T0 and mean MAP over 12 h). Nothing is removed here.

In [ ]:

print("=" * 70); print("C4. CORRELACIÓN ENTRE VARIABLES CLÍNICAS (SPEARMAN)"); print("=" * 70)
vars_c = [c for c in CLIN_MODELO if dfv[c].notna().sum() > 5]
corr = dfv[vars_c].corr(method='spearman')
plt.figure(figsize=(11, 9))
sns.heatmap(corr.rename(index=NOMBRES, columns=NOMBRES), mask=np.triu(np.ones_like(corr, dtype=bool)),
            annot=True, fmt='.2f', cmap='coolwarm', center=0, vmin=-1, vmax=1, cbar_kws={'shrink': .7})
plt.title('Correlación de Spearman entre variables clínicas'); plt.tight_layout()
plt.savefig(f'{RUTA_EDA}/C4_correlacion.png', dpi=150); plt.show()

pares = corr.where(np.triu(np.ones(corr.shape), k=1).astype(bool)).stack().reset_index()
pares.columns = ['var_1', 'var_2', 'rho']
pares = pares.reindex(pares['rho'].abs().sort_values(ascending=False).index).reset_index(drop=True)
print("Top 10 pares:"); display(pares.head(10).round(3))
print(f"Pares con |ρ| ≥ 0,9: {(pares['rho'].abs() >= .9).sum()}")


## C5. Included versus excluded windows

Three selection-bias comparisons (SMD):
1. Windows with >= 6 h of HRV versus 1-5 h.
2. Within the analytic cohort: SOFA outcome evaluable versus not evaluable.
3. Within the analytic cohort: haemodynamic outcome evaluable versus not evaluable (requires measured lactate).

In [ ]:

print("=" * 70); print("C5. INCLUIDAS vs EXCLUIDAS (SMD)"); print("=" * 70)
inc = ult[ult['n_horas_validas'] >= UMBRAL_HRV]
exc = ult[(ult['n_horas_validas'] > 0) & (ult['n_horas_validas'] < UMBRAL_HRV)]
ev_s, no_s = dfv[dfv['label_sofa_responder'].notna()], dfv[dfv['label_sofa_responder'].isna()]
ev_c, no_c = dfv[dfv['label_cardio'].notna()], dfv[dfv['label_cardio'].isna()]
filas = []
for c in CLIN_MODELO + ['vasopresor_activo', 'lactato_medido_12h']:
    b = c in BINARIAS
    filas.append({'variable': NOMBRES.get(c, c),
                  f'≥{UMBRAL_HRV}h vs 1–{UMBRAL_HRV - 1}h': smd(inc[c], exc[c], b),
                  'SOFA evaluable vs no': smd(ev_s[c], no_s[c], b),
                  'Hemodinámico evaluable vs no': smd(ev_c[c], no_c[c], b)})
t5 = pd.DataFrame(filas).set_index('variable').round(3)
print(f"n: ≥{UMBRAL_HRV}h={len(inc)} | 1–{UMBRAL_HRV - 1}h={len(exc)} | SOFA evaluable={len(ev_s)} / no={len(no_s)} | hemodinámico evaluable={len(ev_c)} / no={len(no_c)}")
display(t5.style.map(colorear_smd))
t5.to_csv(f'{RUTA_EDA}/C5_smd_incluidas_excluidas.csv')


## C6. Clinical variables by outcome, compared with HRV

SMD between classes (class 1 = improvement / stability or recovery / deterioration). The largest clinical |SMD| is compared with the largest HRV |SMD| from the HRV EDA. **Not variable selection.**

In [ ]:

print("=" * 70); print("C6. CLÍNICA POR DESENLACE (SMD)"); print("=" * 70)
mat = {}
for lab, (nom, _) in LABELS.items():
    s = dfv[dfv[lab].notna()]
    mat[nom] = {NOMBRES.get(c, c): smd(s.loc[s[lab] == 1, c], s.loc[s[lab] == 0, c], c in BINARIAS)
                for c in CLIN_MODELO + ['vasopresor_activo', 'lactato_medido_12h']}
    print(f"{nom}: {len(s)} ventanas | clase 1 = {int(s[lab].sum())} ({100 * s[lab].mean():.1f} %)")
mat = pd.DataFrame(mat); mat.to_csv(f'{RUTA_EDA}/C6_smd_clinica_por_desenlace.csv')
lim = max(.3, np.nanmax(np.abs(mat.values)))
plt.figure(figsize=(8, 8))
sns.heatmap(mat, annot=True, fmt='.2f', cmap='coolwarm', center=0, vmin=-lim, vmax=lim, cbar_kws={'label': 'SMD'})
plt.title('Variables clínicas: clase 1 − clase 0'); plt.tight_layout()
plt.savefig(f'{RUTA_EDA}/C6_smd_heatmap.png', dpi=150); plt.show()

comp = pd.DataFrame({'Máx |SMD| clínica': mat.abs().max(), 'Variable clínica': mat.abs().idxmax()})
if os.path.exists(RUTA_SMD_HRV):
    hrv_smd = pd.read_csv(RUTA_SMD_HRV, index_col=0)
    mapa = {'SOFA respuesta (principal)': 'SOFA respuesta (principal)',
            'Cardiovascular (principal)': 'Hemodinámico (principal)',
            'SOFA deterioro ≥2 (secundario)': 'SOFA progresión ≥2 (secundario)'}
    hrv_smd = hrv_smd.rename(columns=mapa)
    comp['Máx |SMD| HRV'] = hrv_smd.abs().max().reindex(comp.index)
    comp['Variable HRV'] = hrv_smd.abs().idxmax().reindex(comp.index)
else:
    print("(No se encontró B6_smd_por_label.csv del EDA de HRV: corre primero ese notebook para la comparación.)")
print("\nMayor separación univariada por desenlace:"); display(comp.round(3))


## C7. SOFA outcome: components, delta SOFA and new laboratory values

Which components are measured at T0, the distribution of the paired delta SOFA, and windows with versus without a new laboratory value in the horizon (all windows are used in the main analysis; the restriction is a sensitivity analysis).

In [ ]:

print("=" * 70); print("C7. DESENLACE SOFA"); print("=" * 70)
s = dfv[dfv['label_sofa_responder'].notna()].copy()

comp_t0 = pd.DataFrame({c.replace('sofa_', '').replace('_t0', ''): s[c].value_counts(dropna=False).sort_index() for c in COMPONENTES}).fillna(0).astype(int)
comp_t0.index = comp_t0.index.map(lambda x: 'sin dato' if pd.isna(x) else int(x))
print("Puntaje de cada componente en T0 (ventanas SOFA evaluables):"); display(comp_t0)

d = s.loc[s['muere_en_horizonte'] == 0, 'delta_sofa_pareado']
print(f"\nΔSOFA pareado (sin muertes forzadas): ΔSOFA<0 {100 * (d < 0).mean():.1f} % | =0 {100 * (d == 0).mean():.1f} % | >0 {100 * (d > 0).mean():.1f} % | ≥2 {100 * (d >= 2).mean():.1f} %")
plt.figure(figsize=(9, 3.8))
sns.countplot(x=d.astype(int), color='teal')
plt.title('Distribución del ΔSOFA pareado (T0 + 6 h − T0)'); plt.xlabel('ΔSOFA'); plt.ylabel('Ventanas')
plt.tight_layout(); plt.savefig(f'{RUTA_EDA}/C7_delta_sofa.png', dpi=150); plt.show()

s2 = s[s['muere_en_horizonte'] == 0].copy()
s2['grupo'] = s2['new_lab_6h'].fillna(0).map({1: 'Con laboratorio nuevo', 0: 'Sin laboratorio nuevo'})
t7 = s2.groupby('grupo').agg(
    ventanas=('icustay_id', 'size'), pacientes=('icustay_id', 'nunique'),
    pct_delta_cero=('delta_sofa_pareado', lambda x: round(100 * (x == 0).mean(), 1)),
    pct_responder=('label_sofa_responder', lambda x: round(100 * x.mean(), 1)),
    pct_progresion=('label_sofa_deterioro_2pts', lambda x: round(100 * x.mean(), 1)),
    sofa_t0_mediana=('sofa_t0_parcial', 'median'),
    pct_vasopresor=('vasopresor_activo', lambda x: round(100 * x.mean(), 1)))
print("\nCon y sin laboratorio nuevo en el horizonte:"); display(t7)
t7.to_csv(f'{RUTA_EDA}/C7_laboratorio_nuevo.csv')


## C8. Haemodynamic outcome: which criteria are met

Splits the outcome into its MAP and lactate parts and shows the lactate category transitions between the observation window and the horizon.

In [ ]:

print("=" * 70); print("C8. DESENLACE HEMODINÁMICO"); print("=" * 70)
h = dfv[dfv['label_cardio'].notna() & (dfv['muere_en_horizonte'] == 0)].copy()
pam1 = h['mbp_pct_next6h_gt65'] > 0.5
pam2 = h['mbp_incremento_pct'] >= 0.10
la, ln = h['lactate_last_actual12h'], h['lactate_last_next6h']
lac1 = (la < 2) & (ln < 2); lac2 = la.between(2, 4) & (ln < 2); lac3 = (la > 4) & (ln <= 4)
resumen = pd.Series({
    'PAM > 65 en > 50 % del horizonte': pam1.mean(),
    'PAM media del horizonte ≥ 10 % sobre la observación': pam2.mean(),
    'Algún criterio de PAM': (pam1 | pam2).mean(),
    'Lactato normal sostenido': lac1.mean(),
    'Lactato leve → normal': lac2.mean(),
    'Lactato severo → leve o normal': lac3.mean(),
    'Algún criterio de lactato': (lac1 | lac2 | lac3).mean(),
    'Respuesta hemodinámica (PAM y lactato)': h['label_cardio'].mean()}).mul(100).round(1)
print(f"{len(h)} ventanas evaluables (sin muertes forzadas)"); display(resumen.to_frame('% de ventanas'))

cat = lambda x: pd.cut(x, [-np.inf, 2, 4, np.inf], labels=['normal (<2)', 'leve (2–4)', 'severo (>4)'], right=False)
print("Transición de lactato (filas: observación; columnas: horizonte), % por fila:")
display(pd.crosstab(cat(la), cat(ln), normalize='index').mul(100).round(1))


## C9. Clinical trajectories by T0 hour

Mean of selected clinical variables by T0 hour, split by SOFA-outcome class. One window per patient per T0 hour; hours with fewer than 5 windows per class are omitted.

In [ ]:

print("=" * 70); print("C9. TRAYECTORIAS CLÍNICAS"); print("=" * 70)
lab = 'label_sofa_responder'; nom, mapa = LABELS[lab]
s = dfv[dfv[lab].notna()].copy(); s['Clase'] = s[lab].map(mapa)
ct = pd.crosstab(s['hour_index'], s[lab])
horas_ok = ct[(ct.get(0.0, 0) >= 5) & (ct.get(1.0, 0) >= 5)].index
s = s[s['hour_index'].isin(horas_ok)]
cols = ['sofa_t0_parcial', 'map_t0', 'vasopresor_activo', 'lactate_last_actual12h', 'sedante_bin', 'spo2_t0']
d = s.melt(id_vars=['hour_index', 'Clase'], value_vars=cols, var_name='Variable', value_name='Valor')
d['Variable'] = d['Variable'].map(lambda c: NOMBRES.get(c, c))
g = sns.relplot(data=d, x='hour_index', y='Valor', hue='Clase', col='Variable', col_wrap=3, kind='line',
                marker='o', errorbar=('ci', 95), facet_kws={'sharey': False}, height=3.3, aspect=1.3, palette='Set2')
g.set_axis_labels('Hora del T0 desde el inicio de la sepsis', 'Media')
g.figure.suptitle(f'Trayectorias clínicas por clase — {nom}', y=1.03)
plt.savefig(f'{RUTA_EDA}/C9_trayectorias.png', dpi=150, bbox_inches='tight'); plt.show()


## C10. Events per variable (overfitting risk)

EPV = windows in the minority class / number of variables in the set. Values below ~10 indicate overfitting risk for unpenalised models and motivate regularisation. Windows of the same patient are not independent, so the effective EPV is lower.

In [ ]:

print("=" * 70); print("C10. EVENTOS POR VARIABLE"); print("=" * 70)
hrv_all = COLS['features_hrv']
conjuntos = {
    'HRV completa': hrv_all,
    'HRV lineal': COLS['features_hrv_lineal'],
    'HRV no lineal': COLS['features_hrv_nolineal'],
    'Clínico basal': COLS['features_clinicas'] + PAMLAC,
    'Clínico basal + SOFA': COLS['features_clinicas'] + PAMLAC + SOFA_F,
    'HRV integrada': hrv_all + COLS['features_clinicas'] + PAMLAC,
    'HRV integrada + SOFA': hrv_all + COLS['features_clinicas'] + PAMLAC + SOFA_F,
}
filas = []
for lab, (nom, _) in LABELS.items():
    s = dfv[dfv[lab].notna()]
    minoritaria = int(min(s[lab].sum(), (s[lab] == 0).sum()))
    pac_evento = s.loc[s[lab] == 1, 'icustay_id'].nunique()
    for cj, vs in conjuntos.items():
        filas.append({'Desenlace': nom, 'Conjunto': cj, 'Variables': len(vs),
                      'Eventos (clase minoritaria)': minoritaria, 'Pacientes con evento': pac_evento,
                      'EPV': round(minoritaria / len(vs), 1)})
epv = pd.DataFrame(filas)
display(epv.pivot(index='Conjunto', columns='Desenlace', values='EPV').reindex(list(conjuntos)))
epv.to_csv(f'{RUTA_EDA}/C10_epv.csv', index=False)


---
## Reading guide

| Section | What to look at | Used for |
|---|---|---|
| C1 | Table 1 | Cohort description |
| C2 | Variables with most missing data (expected: lactate) | Imputation inside the fold; lactate-measured indicator |
| C4 | Redundant pairs | Interpretation of coefficients and SHAP |
| C5 | Clinical selection bias | Limitations |
| C6 | Clinical versus HRV separation | Realistic expectation for H1 |
| C7, C8 | How the outcomes are built | Outcome discussion and limitations |
| C10 | EPV per feature set | Justifies penalised models |

In [ ]:

print("Archivos generados en", RUTA_EDA)
for f in sorted(os.listdir(RUTA_EDA)):
    print("  -", f)
